In [1]:
import pandas as pd
import config
import torch
import torch.nn as nn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
def split_by_machine(df, train_ratio=0.8):
    train_list, test_list = [], []
    for id, data in df.groupby('machineID'):
        data = data.sort_values('datetime')
        split_idx = int(len(data) * train_ratio)
        train_list.append(data.iloc[:split_idx])
        test_list.append(data.iloc[split_idx:])
    return pd.concat(train_list), pd.concat(test_list)

def create_sequences(data, seq_len=7):
    input, target = [], []
    for machine_id in data['machineID'].unique():
        machine_data = data[data['machineID'] == machine_id].sort_values('datetime')

In [3]:
# 加载数据集
errors = pd.read_csv(config.RAW_DATA_DIR / "PdM_errors.csv")
failures = pd.read_csv(config.RAW_DATA_DIR / "PdM_failures.csv")
machines = pd.read_csv(config.RAW_DATA_DIR / "PdM_machines.csv")
maint = pd.read_csv(config.RAW_DATA_DIR / "PdM_maint.csv")
telemetry = pd.read_csv(config.RAW_DATA_DIR / "PdM_telemetry.csv")

In [4]:
# 日期格式
date_tables = [errors, failures, maint, telemetry]
for df in date_tables:
    df['datetime'] = pd.to_datetime(df['datetime'], format="%Y-%m-%d %H:%M:%S")
    df.sort_values(['machineID', 'datetime'], inplace=True, ignore_index=True)

In [5]:
# 合并数据集——machines
dataset = pd.merge(left=machines, right=telemetry,
                    on=['machineID']).sort_values(['datetime', 'machineID'])

# 合并数据集——errors
errors.sort_values(['datetime', 'machineID'], inplace=True)
dataset = pd.merge(left=dataset, right=errors,
                    how='left',
                    on=['datetime', 'machineID'])

# 合并数据集——maint
maint.sort_values(['datetime','machineID'], inplace=True)
dataset = pd.merge(left=dataset, right=maint,
                    how='left',
                    on=['datetime', 'machineID'])

# 合并数据集——failures（计算RUL）
failures['RUL'] = failures['datetime']
failures.sort_values(['datetime', 'machineID'], inplace=True)
dataset = pd.merge_asof(left=dataset, right=failures[['datetime', 'machineID', 'RUL']],
                        on='datetime',
                        by='machineID',
                        direction='forward').sort_values(['machineID','datetime']).reset_index(drop=True)
dataset['RUL'] = (dataset['RUL'] - dataset['datetime']).dt.total_seconds() / (24 * 3600)

In [6]:
# 填补数据集
dataset['RUL'] = dataset['RUL'].fillna(365)

In [11]:
# 划分数据集
train, test = split_by_machine(dataset)

In [12]:
# 特征预处理
other_features = ['machineID', 'datetime', 'RUL', 'age']
numerical_features = ['volt', 'rotate', 'pressure', 'vibration']
categorical_features = ['errorID', 'comp']

numerical_transformer = Pipeline(
    steps=[
        ("fillna", SimpleImputer(strategy='mean')),
        ("std", StandardScaler())
        ]
    )

categorical_transformer = Pipeline(
    steps=[
        ("fillna", SimpleImputer(strategy="constant", fill_value="NaN")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]
    )

preprocessor = ColumnTransformer(
    transformers=[
        ("other", "passthrough", other_features),        
        ("model", OneHotEncoder(drop="first"), ["model"]),        
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
        ]
    )


preprocessor.fit(train)
train = pd.DataFrame(preprocessor.transform(train), columns=preprocessor.get_feature_names_out())
test = pd.DataFrame(preprocessor.transform(test), columns=preprocessor.get_feature_names_out())

In [ ]:
# 将预处理后的2D数据重塑为3D数据
def create_rolling_window(data, seq_len=7):
    """
    data:(batch_size*seq_len + machineID + datetime + RUL, feature_num)
    输出
    input:(batch_size, seq_len, feature_num)
    target:(batch_size, seq_len)
    """

In [ ]:
model = nn.Sequential(
    nn.LSTM(input_size=feature_num,
            hidden_size=,
            num_layers=1,
            batch_first=True)
    )

model(input, target)